# (부록) A2A 프로토콜 멀티 에이전트 (Agent-to-Agent Multi-Agent)
## `a2a-sdk` 로 여러 에이전트가 통신하며 일을 분담하기

> 📦 **환경 설치·실행 명령**은 [`env_guides/M02_5_a2a_multiagent.md`](env_guides/M02_5_a2a_multiagent.md) 에 정리되어 있습니다.
> 이 노트북은 [`M02_3_mcp_a2a.ipynb`](M02_3_mcp_a2a.ipynb) 에서 개념적으로 다룬 A2A 를,
> 실제 표준 구현체인 **`a2a-sdk`(a2a-python) 1.1.0** 으로 확장합니다.

---

### A2A 프로토콜이란?
**A2A(Agent-to-Agent)** 는 서로 다른 프레임워크·벤더로 만든 에이전트들이 **상호 운용**하도록
Google 이 공개한 개방형 프로토콜입니다. 핵심 개념은 다음과 같습니다.

- **AgentCard** — 에이전트의 "명함". 이름·설명·버전, 제공 **스킬(AgentSkill)**, **역량(AgentCapabilities,
  스트리밍 등)**, 접속 엔드포인트(**AgentInterface**: URL + 전송 바인딩)를 담는다.
  `/.well-known/agent-card.json` 경로로 **발견(discovery)** 된다.
- **Message / Part** — 에이전트 간 주고받는 메시지. 텍스트·데이터·파일 등 여러 **Part** 로 구성.
- **Task** — 하나의 작업 단위. 상태(`submitted → working → completed/failed`)와 결과 **Artifact** 를 가진다.
- **전송 바인딩** — JSON-RPC / gRPC / HTTP+JSON. 클라이언트는 카드를 보고 지원 바인딩으로 접속한다.

### 이 노트북이 보여줄 것
1. **AgentCard/AgentSkill** 로 전문 에이전트 3개(계산·요약·작문) 정의
2. 각 에이전트를 **실제 A2A HTTP 서버**로 (백그라운드 스레드에) 기동
3. `A2ACardResolver` 로 **카드 발견**, `ClientFactory` 로 **A2A 클라이언트** 생성
4. **코디네이터**가 복합 요청을 LLM 으로 **분해**하고, 각 전문 에이전트에 **A2A 메시지로 위임**한 뒤,
   작문 에이전트로 **결과를 취합**

### 전제조건
- Windows 11 + **CMD(`cmd.exe`)** + Python **3.11**(`uv` 관리), 커널 = **`Agentic AI (uv)`**
- 기본 LLM = **로컬 Ollama + `qwen3:8b`**. 각 전문 에이전트의 두뇌로 이 LLM 을 사용합니다.
- `a2a-sdk` / `uvicorn` / `httpx` 필요(아래 setup 셀이 자동 설치).

### 아키텍처 개요
```
[사용자 복합 요청]
        │
   [Coordinator] ── LLM 으로 하위작업 분해
        │  A2A 메시지(JSON-RPC over HTTP)
        ├──▶ [math_agent  : A2A HTTP 서버]  (Ollama)
        ├──▶ [research_agent: A2A HTTP 서버] (Ollama)
        └──▶ [writer_agent  : A2A HTTP 서버] (Ollama) ── 결과 취합
```


In [1]:
# [setup] 자기완결적 셋업 — 이 노트북만 단독 실행 가능하게 함.
import sys, os
sys.path.insert(0, os.path.abspath(''))   # notebooks/ 를 import 경로에 추가

import utils
utils.reload_env()   # .env 재로드 (LLM_PROVIDER 등 갱신)

from agentic_lib import bootstrap
from agentic_lib.bootstrap import to_text   # 공급자 무관 응답 정규화(<think> 제거 포함)

# A2A 실습에 필요한 패키지 (A2A 표준 SDK + ASGI 서버 + 비동기 HTTP 클라이언트)
utils.uv_install(['a2a-sdk', 'uvicorn', 'httpx',
                  'langchain', 'langchain-openai'])

llm = utils.get_llm()   # 각 전문 에이전트의 두뇌로 사용할 LLM (기본 ollama/qwen3:8b)
print("준비 완료:", utils.LLM_PROVIDER)


LLM 공급자: nvidia
  NVIDIA build Key: 설정됨  /  Model: meta/llama-3.1-8b-instruct
[uv] 설치 완료: ['a2a-sdk', 'uvicorn', 'httpx', 'langchain', 'langchain-openai']


준비 완료: nvidia


---
## 1. A2A SDK 임포트와 공통 헬퍼

`a2a-sdk` 1.1.0 의 실제 심볼을 사용합니다(버전마다 API 가 크게 다르므로 설치된 버전 기준).

| 심볼 | 역할 |
|---|---|
| `AgentCard`, `AgentSkill`, `AgentCapabilities`, `AgentInterface` | 에이전트 명세(발견용) |
| `Message`, `Part`, `Role`, `Task`, `TaskState` | 메시지·작업 타입(프로토콜 버퍼 기반) |
| `AgentExecutor`, `RequestContext`, `EventQueue` | **서버 측** 에이전트 실행 모델 |
| `TaskUpdater` | 작업 상태·결과(Artifact)를 이벤트 큐에 기록하는 헬퍼 |
| `DefaultRequestHandler`, `InMemoryTaskStore` | 요청 처리기 + 작업 저장소 |
| `create_agent_card_routes`, `create_jsonrpc_routes` | Starlette 라우트 생성(→ `uvicorn` 서빙) |
| `ClientFactory`, `ClientConfig`, `A2ACardResolver` | **클라이언트 측** 카드 발견 + 접속 |
| `new_text_message`, `new_text_part`, `new_task`, `get_stream_response_text` | 메시지/작업 생성·파싱 헬퍼 |

아래 셀은 임포트와, 뒤에서 재사용할 작은 헬퍼(`free_port`, `make_agent_card`, `LLMAgentExecutor`)를 정의합니다.
`LLMAgentExecutor` 가 A2A 서버가 요청을 받을 때마다 호출하는 **에이전트 본체**로, 역할별 시스템 프롬프트로 로컬 LLM 을 구동합니다.


In [2]:
import asyncio, threading, time, socket, json, re
import httpx, uvicorn
from starlette.applications import Starlette
from langchain_core.messages import SystemMessage, HumanMessage

# --- a2a-sdk 1.1.0 실제 심볼 ---
from a2a.types import (
    AgentCard, AgentSkill, AgentCapabilities, AgentInterface,
    Role, SendMessageRequest, TaskState,
)
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.tasks import InMemoryTaskStore
from a2a.server.tasks.task_updater import TaskUpdater
from a2a.server.routes import create_agent_card_routes, create_jsonrpc_routes
from a2a.client import ClientFactory, ClientConfig, A2ACardResolver
from a2a.utils import TransportProtocol
from a2a.helpers.proto_helpers import (
    new_text_message, new_text_part, new_task, get_stream_response_text,
)


def free_port() -> int:
    '''사용 가능한 임의의 로컬 포트를 반환한다(포트 충돌 방지).'''
    s = socket.socket()
    s.bind(("127.0.0.1", 0))
    port = s.getsockname()[1]
    s.close()
    return port


class LLMAgentExecutor(AgentExecutor):
    '''역할별 시스템 프롬프트로 로컬 LLM 을 구동하는 A2A 서버 측 실행기.

    A2A 서버는 요청 1건마다 ``execute(context, event_queue)`` 를 호출한다.
    여기서 사용자 입력을 꺼내 LLM 에 넘기고, 결과를 ``TaskUpdater`` 로
    Task 의 Artifact(결과물)에 기록한 뒤 완료 상태로 만든다.
    '''

    def __init__(self, llm, system_prompt: str):
        self.llm = llm                      # 이 에이전트의 두뇌(LangChain 모델)
        self.system_prompt = system_prompt  # 역할 정의(계산/요약/작문 등)

    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        user_text = context.get_user_input()          # 들어온 Message 의 텍스트
        # 새 요청이면 Task 를 먼저 이벤트 큐에 넣어야 한다(상태 이벤트보다 선행 필수)
        if context.current_task is None:
            await event_queue.enqueue_event(
                new_task(context.task_id, context.context_id, TaskState.TASK_STATE_SUBMITTED)
            )
        updater = TaskUpdater(event_queue, context.task_id, context.context_id)
        await updater.start_work()                     # 상태: working
        # LLM 호출은 동기 함수이므로 이벤트 루프를 막지 않게 스레드로 offload
        answer = await asyncio.to_thread(
            bootstrap.invoke_text, self.llm,
            [SystemMessage(self.system_prompt), HumanMessage(user_text)],
        )
        await updater.add_artifact([new_text_part(answer)], name="result")  # 결과물 첨부
        await updater.complete(                          # 상태: completed
            message=updater.new_agent_message([new_text_part(answer)])
        )

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        '''취소는 이 예제에서 지원하지 않는다.'''
        raise NotImplementedError("cancel is not supported in this demo")


print("A2A 심볼 임포트 완료. TransportProtocol.JSONRPC =", TransportProtocol.JSONRPC.value)


A2A 심볼 임포트 완료. TransportProtocol.JSONRPC = JSONRPC


---
## 2. 전문 에이전트 정의 (AgentCard · AgentSkill)

각 에이전트는 자신을 설명하는 **AgentCard** 를 게시합니다. 카드에는 제공 **스킬(AgentSkill)**,
**역량(AgentCapabilities)**, 접속 **인터페이스(AgentInterface: URL + 전송 바인딩)** 가 담깁니다.
클라이언트/코디네이터는 이 카드를 **발견**해 무엇을 할 수 있는 에이전트인지 파악하고 접속합니다.

여기서는 세 전문 에이전트를 둡니다.

| 에이전트 | 역할(시스템 프롬프트) | 스킬 |
|---|---|---|
| `math_agent` | 정확한 산술 계산 | `calc` |
| `research_agent` | 주제를 짧게 요약 설명 | `summarize` |
| `writer_agent` | 자료를 한 단락으로 정리 | `compose` |

`make_agent_card()` 는 포트에 맞춘 JSON-RPC 인터페이스를 가진 카드를 만들어 줍니다.


In [3]:
def make_agent_card(name, description, skill_id, skill_name, port) -> AgentCard:
    '''JSON-RPC 인터페이스를 가진 전문 에이전트용 AgentCard 를 생성한다.'''
    return AgentCard(
        name=name,
        description=description,
        version="1.0.0",
        # 접속 지점 + 전송 바인딩(JSON-RPC). 클라이언트는 이 정보로 접속한다.
        supported_interfaces=[AgentInterface(
            url=f"http://127.0.0.1:{port}/",
            protocol_binding=TransportProtocol.JSONRPC.value,
            protocol_version="1.0",
        )],
        capabilities=AgentCapabilities(streaming=True),   # 역량: 스트리밍 지원
        default_input_modes=["text/plain"],
        default_output_modes=["text/plain"],
        skills=[AgentSkill(id=skill_id, name=skill_name,
                           description=description, tags=[skill_id])],
    )


# (역할 프롬프트, 스킬 id, 스킬 이름) — 이름이 곧 A2A 상 에이전트 식별자
AGENT_SPECS = {
    "math_agent":     ("너는 계산 전문 에이전트다. 산술 계산만 정확히 수행하고 최종 숫자만 간결히 답한다.",
                       "calc", "계산"),
    "research_agent": ("너는 조사 전문 에이전트다. 주어진 주제를 2문장 이내로 쉽게 요약 설명한다.",
                       "summarize", "요약"),
    "writer_agent":   ("너는 작문 전문 에이전트다. 입력으로 받은 자료들을 자연스러운 한 단락으로 정리한다.",
                       "compose", "작문"),
}

# 카드 + 실행기(LLM 두뇌) 생성
agent_cards = {}
agent_executors = {}
for _name, (_prompt, _sid, _sname) in AGENT_SPECS.items():
    _port = free_port()
    agent_cards[_name] = make_agent_card(_name, _prompt, _sid, _sname, _port)
    agent_executors[_name] = LLMAgentExecutor(llm, _prompt)

for _name, _card in agent_cards.items():
    print(f"{_name:15s} -> {_card.supported_interfaces[0].url}  skills={[s.name for s in _card.skills]}")


math_agent      -> http://127.0.0.1:51789/  skills=['계산']
research_agent  -> http://127.0.0.1:51790/  skills=['요약']
writer_agent    -> http://127.0.0.1:51791/  skills=['작문']


---
## 3. 각 에이전트를 실제 A2A HTTP 서버로 기동

A2A 서버는 다음으로 구성됩니다.

1. `DefaultRequestHandler(agent_executor, task_store, agent_card)` — 들어온 요청을 실행기로 넘기고 작업을 저장
2. `create_agent_card_routes(card)` + `create_jsonrpc_routes(handler, rpc_url="/")` — Starlette 라우트
3. `Starlette(routes=...)` 앱을 `uvicorn` 으로 서빙

노트북에서 셀을 막지 않도록 각 서버를 **백그라운드 스레드**에서 실행합니다(운영에서는 각 에이전트가
독립 서비스/컨테이너로 뜹니다). `server.started` 로 준비 상태를 기다린 뒤 다음 셀로 진행합니다.


In [4]:
def start_a2a_server(card: AgentCard, executor: AgentExecutor):
    '''AgentCard/실행기로 A2A HTTP 서버를 백그라운드 스레드에서 띄우고 (server, thread) 반환.'''
    handler = DefaultRequestHandler(
        agent_executor=executor,
        task_store=InMemoryTaskStore(),   # 데모용 인메모리 작업 저장소
        agent_card=card,
    )
    # 카드 발견 라우트(/.well-known/agent-card.json) + JSON-RPC 라우트(/)
    routes = create_agent_card_routes(card) + create_jsonrpc_routes(handler, rpc_url="/")
    app = Starlette(routes=routes)

    port = int(card.supported_interfaces[0].url.rstrip("/").split(":")[-1])
    config = uvicorn.Config(app, host="127.0.0.1", port=port, log_level="warning")
    server = uvicorn.Server(config)
    thread = threading.Thread(target=server.run, daemon=True)  # 데몬 스레드로 실행
    thread.start()
    for _ in range(50):          # 서버가 소켓을 열 때까지 최대 ~5초 대기
        if server.started:
            break
        time.sleep(0.1)
    return server, thread


# 세 에이전트 서버 기동
running_servers = {}
for _name in agent_cards:
    _srv, _th = start_a2a_server(agent_cards[_name], agent_executors[_name])
    running_servers[_name] = (_srv, _th)
    print(f"{_name:15s} started={_srv.started}  {agent_cards[_name].supported_interfaces[0].url}")


math_agent      started=True  http://127.0.0.1:51789/


research_agent  started=True  http://127.0.0.1:51790/


writer_agent    started=True  http://127.0.0.1:51791/


---
## 4. A2A 카드 발견 (discovery)

클라이언트는 에이전트 URL 만 알면 `A2ACardResolver` 로 `/.well-known/agent-card.json` 을 읽어
그 에이전트의 **AgentCard** 를 가져올 수 있습니다. 이것이 A2A 의 **발견(discovery)** 입니다.
아래에서 실제 HTTP 로 카드를 가져와 이름·스킬·전송 바인딩을 확인합니다.

> 이 노트북은 비동기 A2A API 를 사용하므로, 셀에서 **최상위 `await`** 를 사용합니다(Jupyter 지원).


In [5]:
# 여러 셀에서 재사용할 단일 비동기 HTTP 클라이언트
http_client = httpx.AsyncClient(timeout=120)

# 각 에이전트의 URL 만 알고 있다고 가정하고, 하나씩 카드를 발견해 본다.
# 실제로는 서로 다른 호스트/서비스일 수 있으므로 URL 목록만으로 발견을 반복한다.
discovered_cards = {}
for _name, _card in agent_cards.items():
    _url = _card.supported_interfaces[0].url.rstrip("/")
    resolver = A2ACardResolver(http_client, base_url=_url)
    discovered = await resolver.get_agent_card()   # /.well-known/agent-card.json 발견
    discovered_cards[_name] = discovered

# 발견한 모든 카드의 정보를 출력
for _name, discovered in discovered_cards.items():
    print("=" * 60)
    print("발견한 카드 이름 :", discovered.name)
    print("설명            :", discovered.description)
    print("스킬            :", [(s.id, s.name) for s in discovered.skills])
    print("전송 바인딩      :", [i.protocol_binding for i in discovered.supported_interfaces])
    print("스트리밍 지원    :", discovered.capabilities.streaming)
print("=" * 60)
print(f"총 {len(discovered_cards)}개 에이전트 카드 발견 완료")


발견한 카드 이름 : math_agent
설명            : 너는 계산 전문 에이전트다. 산술 계산만 정확히 수행하고 최종 숫자만 간결히 답한다.
스킬            : [('calc', '계산')]
전송 바인딩      : ['JSONRPC']
스트리밍 지원    : True
발견한 카드 이름 : research_agent
설명            : 너는 조사 전문 에이전트다. 주어진 주제를 2문장 이내로 쉽게 요약 설명한다.
스킬            : [('summarize', '요약')]
전송 바인딩      : ['JSONRPC']
스트리밍 지원    : True
발견한 카드 이름 : writer_agent
설명            : 너는 작문 전문 에이전트다. 입력으로 받은 자료들을 자연스러운 한 단락으로 정리한다.
스킬            : [('compose', '작문')]
전송 바인딩      : ['JSONRPC']
스트리밍 지원    : True
총 3개 에이전트 카드 발견 완료


---
## 5. A2A 클라이언트로 단일 에이전트 호출

`ClientFactory(ClientConfig(...))` 로 카드에 맞는 클라이언트를 만들고, `SendMessageRequest` 에
`Message` 를 실어 보냅니다. 응답은 `StreamResponse` 스트림으로 오며(여기서는 `streaming=False`
설정이라 완료된 Task 로 옴), `get_stream_response_text()` 로 텍스트를 뽑습니다.

`ask_agent()` 는 뒤의 코디네이터가 각 전문 에이전트에 **작업을 위임**할 때 재사용하는 헬퍼입니다.


In [6]:
async def ask_agent(card: AgentCard, text: str) -> str:
    '''주어진 AgentCard 의 에이전트에게 A2A 메시지를 보내고 응답 텍스트를 반환한다.'''
    config = ClientConfig(
        httpx_client=http_client,
        streaming=False,                                        # 완료된 Task 로 수신
        supported_protocol_bindings=[TransportProtocol.JSONRPC.value],
    )
    client = ClientFactory(config).create(card)                 # 카드 → A2A 클라이언트
    request = SendMessageRequest(message=new_text_message(text, role=Role.ROLE_USER))
    parts = []
    async for response in client.send_message(request):         # 응답 스트림 소비
        chunk = get_stream_response_text(response)
        if chunk:
            parts.append(chunk)
    return "\n".join(parts)


# 단일 에이전트 호출 데모: 계산 에이전트에 A2A 메시지 전송
_answer = await ask_agent(agent_cards["math_agent"], "3200 곱하기 4는 얼마인가?")
print("math_agent 응답 :", _answer)


math_agent 응답 : 12800


---
## 6. 코디네이터: 복합 요청 분해 → A2A 위임 → 취합

이제 여러 에이전트가 **협업**하도록 **코디네이터**를 만듭니다. 코디네이터는

1. 사용자의 **복합 요청**을 LLM 으로 **하위작업(subtask)** 으로 **분해**하고(어떤 전문 에이전트가 처리할지 지정),
2. 각 하위작업을 해당 전문 에이전트에게 **A2A 메시지로 위임**(`ask_agent`)한 뒤,
3. 수집한 결과를 **작문 에이전트(`writer_agent`)** 에게 넘겨 **하나의 답으로 취합**합니다.

분해 결과(JSON)가 형식에 어긋나면 안전하게 **폴백**(모든 전문 에이전트에 원 요청 전달)합니다.


In [7]:
class Coordinator:
    '''복합 요청을 분해해 전문 에이전트들에게 A2A 로 위임하고 결과를 취합하는 조정자.'''

    def __init__(self, llm, cards: dict):
        self.llm = llm
        self.cards = cards
        # writer_agent 는 최종 취합 담당이므로 분해 대상(worker)에서 제외
        self.workers = {n: c for n, c in cards.items() if n != "writer_agent"}

    async def decompose(self, request: str) -> list:
        '''LLM 으로 요청을 [{agent, subtask}, ...] 로 분해한다(견고한 파싱 + 폴백).'''
        roster = "\n".join(f"- {n}: {c.description}" for n, c in self.workers.items())
        sys_prompt = (
            "너는 코디네이터다. 사용자 요청을 아래 전문 에이전트가 처리할 하위작업으로 분해한다.\n"
            + roster
            + "\n반드시 JSON 배열만 출력한다. 각 원소는 {\"agent\": 에이전트이름, \"subtask\": 지시문}."
            " 다른 설명은 쓰지 마라."
        )
        raw = await asyncio.to_thread(
            bootstrap.invoke_text, self.llm,
            [SystemMessage(sys_prompt), HumanMessage(request)],
        )
        try:
            plan = json.loads(re.search(r"\[.*\]", raw, re.S).group(0))
            plan = [s for s in plan if s.get("agent") in self.workers and s.get("subtask")]
            if plan:
                return plan
        except Exception:
            pass
        # 폴백: 형식 파싱 실패 시 모든 전문 에이전트에 원 요청을 그대로 위임
        return [{"agent": n, "subtask": request} for n in self.workers]

    async def run(self, request: str) -> str:
        '''요청 분해 → 각 전문 에이전트에 A2A 위임 → writer_agent 로 취합.'''
        plan = await self.decompose(request)
        print("[분해된 계획]")
        for step in plan:
            print(f"  - {step['agent']}: {step['subtask']}")

        results = []
        for step in plan:                                  # 각 하위작업을 A2A 로 위임
            answer = await ask_agent(self.cards[step["agent"]], step["subtask"])
            results.append((step["agent"], answer))
            print(f"\n[{step['agent']} 응답] {answer}")

        material = "\n".join(f"- {name}: {ans}" for name, ans in results)
        final = await ask_agent(                            # 결과 취합(작문 에이전트)
            self.cards["writer_agent"],
            f"아래 자료를 바탕으로 사용자 질문에 대한 최종 답을 한 단락으로 정리하라.\n"
            f"질문: {request}\n자료:\n{material}",
        )
        return final


coordinator = Coordinator(llm, agent_cards)
user_request = "3200원짜리 커피 4잔의 총액을 계산하고, 에스프레소가 무엇인지 설명해줘."
final_answer = await coordinator.run(user_request)
print("\n" + "=" * 60)
print("[코디네이터 최종 답]")
print(final_answer)


[분해된 계획]
  - math_agent: 3200 * 4 계산
  - research_agent: 에스프레소 설명



[math_agent 응답] 12800



[research_agent 응답] 에스프레소는 이탈리아에서 유래한 커피의 한 종류로, 강한 향과 풍부한 맛을 특징으로 합니다. 에스프레소는 고온의 물을 고압으로 강제로 커피콩에 통과시켜 만든 커피로, 일반적으로 1-2ml의 양으로 추출됩니다.



[코디네이터 최종 답]
3200원짜리 커피 4잔의 총액을 계산하면 3200 x 4 = 12800원입니다. 에스프레소는 이탈리아에서 유래한 커피의 한 종류로, 강한 향과 풍부한 맛을 특징으로 하는 커피로, 일반적으로 1-2ml의 양으로 추출됩니다.


---
## 7. 정리 및 서버 종료

실습이 끝나면 HTTP 클라이언트를 닫고 백그라운드 A2A 서버를 종료합니다.


In [8]:
# 비동기 HTTP 클라이언트 종료
await http_client.aclose()

# 백그라운드 A2A 서버 종료 요청(데몬 스레드라 커널 종료 시에도 정리됨)
for _name, (_srv, _th) in running_servers.items():
    _srv.should_exit = True
print("클라이언트/서버 정리 완료")


클라이언트/서버 정리 완료


---
## 8. 마무리 — A2A 핵심과 운영 배포

### 이 노트북에서 실제로 사용한 A2A 요소
- **AgentCard / AgentSkill / AgentCapabilities / AgentInterface** — 에이전트를 자기서술하고 **발견**되게 함
- **A2ACardResolver** — `/.well-known/agent-card.json` 발견
- **AgentExecutor / RequestContext / EventQueue / TaskUpdater** — 서버 측 실행 모델(요청 → Task → Artifact)
- **Message / Task / Artifact** — 요청·작업·결과의 프로토콜 타입
- **ClientFactory / ClientConfig** — 카드 기반 클라이언트 생성, `SendMessageRequest` 로 위임

### 운영 배포 형태
- 노트북에서는 편의상 각 에이전트를 **백그라운드 스레드**로 띄웠지만, 실제로는 각 에이전트가
  **독립 서비스(별도 프로세스/컨테이너/호스트)** 로 뜨고, 코디네이터는 URL 만 알면
  `ClientFactory.create_from_url(url)` 로 카드 발견부터 접속까지 한 번에 처리할 수 있습니다.
- **전송 바인딩**은 JSON-RPC 외에 **gRPC**(`TransportProtocol.GRPC`), **HTTP+JSON**
  (`TransportProtocol.HTTP_JSON`) 도 있으며, 카드의 `supported_interfaces` 로 광고합니다.
- **스트리밍**: `AgentCapabilities(streaming=True)` + `ClientConfig(streaming=True)` 로
  부분 결과를 `TaskStatusUpdateEvent`/`TaskArtifactUpdateEvent` 스트림으로 받을 수 있습니다.

### 참고: URL 만으로 접속하는 코디네이터(운영형) 스케치
```python
# 각 에이전트가 독립 서비스로 떠 있을 때(코드 실행 아님, 형태 예시)
client = await ClientFactory(ClientConfig(httpx_client=hx)).create_from_url(
    "http://math-agent.internal:9001/"   # 카드 발견 + 접속을 한 번에
)
async for resp in client.send_message(
    SendMessageRequest(message=new_text_message("3200*4", role=Role.ROLE_USER))
):
    print(get_stream_response_text(resp))
```

### 한계 / 유의
- 로컬 소형 모델은 분해(JSON) 형식을 자주 어길 수 있어 **폴백 파서**를 두었습니다.
- 인메모리 작업 저장소(`InMemoryTaskStore`)는 데모용입니다. 운영에서는 영속 저장소를 사용합니다.
- 인증/보안(`SecurityScheme`), 푸시 알림, 장기 실행 작업 폴링 등은 SDK 가 지원하지만 이 노트북 범위 밖입니다.
